In [1]:
!pip install 'smolagents[mcp]' firecrawl huggingface_hub

In [8]:
import os
import getpass

os.environ["HF_TOKEN"] = getpass.getpass("Huggingface Token: ")
os.environ["FIRECRAWL_API_KEY"] = getpass.getpass("Firecrawl API Key: ")

Huggingface Token: ··········
Firecrawl API Key: ··········


# Generate Reasearch Plan

In [3]:
PLANNER_SYSTEM_INSTRUCTIONS = """
You will be given a research task by a user. Your job is to produce a set of
instructions for a researcher that will complete the task. Do NOT complete the
task yourself, just provide instructions on how to complete it.

GUIDELINES:
1. Maximize specificity and detail. Include all known user preferences and
   explicitly list key attributes or dimensions to consider.
2. If essential attributes are missing, explicitly state that they are open-ended.
3. Avoid unwarranted assumptions. Treat unspecified dimensions as flexible.
4. Use the first person (from the user's perspective).
5. When helpful, explicitly ask the researcher to include tables.
6. Include the expected output format (e.g. structured report with headers).
7. Preserve the input language unless the user explicitly asks otherwise.
8. Sources: prefer primary / official / original sources.
"""

from huggingface_hub import InferenceClient

def generate_research_plan(user_query: str) -> str:
    MODEL_ID = "moonshotai/Kimi-K2-Thinking"
    PROVIDER = "auto"

    print("Generating the research plan for the query: ", user_query)
    print("MODEL: ", MODEL_ID)
    print("PROVIDER: ", PROVIDER)

    planner_client = InferenceClient(
        api_key=os.environ["HF_TOKEN"],
        provider=PROVIDER,
    )

    completion = planner_client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": PLANNER_SYSTEM_INSTRUCTIONS},
            {"role": "user", "content": user_query},
        ],
    )

    research_plan = completion.choices[0].message.content

    print("\033[93mGenerated Research Plan\033[0m")
    print(f"\033[93m{research_plan}\033[0m")

    return research_plan


research_plan = generate_research_plan("climate in northern france")

Generating the research plan for the query:  climate in northern france
MODEL:  moonshotai/Kimi-K2-Thinking
PROVIDER:  auto


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generated Research Plan
I need a comprehensive climate analysis of northern France. Please conduct research and produce a structured report with the following specifications:

**Geographic Scope**
- Focus on these key regions: Hauts-de-France, Normandy, and Brittany
- Include at minimum these representative cities: Lille, Rouen, Caen, Brest, and Amiens
- If data limitations exist, substitute with the nearest Météo-France official weather station
- *Open-ended*: I have not specified smaller municipalities or microclimates—include these if they show significant variation (>10% difference in precipitation or >1°C in temperature) from regional averages

**Climate Parameters to Investigate**
For each location, provide:
1. **Temperature**: 30-year monthly normals (1991-2020), record extremes (highest/lowest), seasonal averages, frost days per year, and growing season length
2. **Precipitation**: Monthly rainfall totals, number of rainy days (>1mm), seasonal distribution, intensity extremes (

# Split the Plan into Focused Subtasks

In [4]:
import json
from pydantic import BaseModel, Field, validator
from typing import List

TASK_SPLITTER_SYSTEM_INSTRUCTIONS = """
You will be given a set of research instructions (a research plan).
Your job is to break this plan into a set of coherent, non-overlapping
subtasks that can be researched independently by separate agents.

Requirements:
- 3 to 8 subtasks is usually a good range. Use your judgment.
- Each subtask should have:
  - an 'id' (short string),
  - a 'title' (short descriptive title),
  - a 'description' (clear, detailed instructions for the sub-agent).
- Subtasks should collectively cover the full scope of the original plan
  without unnecessary duplication.
- Prefer grouping by dimensions: time periods, regions, actors, themes,
  causal mechanisms, etc., depending on the topic.
- Each description should be very clear and detailed about everything that
  the agent needs to research to cover that topic.
- Do not include a final task that will put everything together.
  This will be done later in another step.

Output format:
Return ONLY valid JSON with this schema:

{
  "subtasks": [
    {
      "id": "string",
      "title": "string",
      "description": "string"
    }
  ]
}
"""

from pprint import pprint

class Subtask(BaseModel):
  id: str = Field(
      ...,
      description="Short identifier for the subtask (e.g. 'A', 'history', 'drivers').",
  )
  title: str = Field(
      ...,
      description="Short descriptive title of the subtask.",
  )
  description: str = Field(
      ...,
      description="Clear, detailed instructions for the sub-agent that will research this subtask.",
  )

class SubtaskList(BaseModel):
  subtasks: List[Subtask] = Field(
      ...,
      description="List of substasks that together cover the whole research plan.",
  )

TASK_SPLITTER_JSON_SCHEMA = {
    "name": "subtaskList",
    "schema": SubtaskList.model_json_schema(),
    "strict": True,
}

def split_into_subtasks(research_plan: str) -> List[dict]:
    MODEL_ID = "openai/gpt-oss-120b"
    PROVIDER = "together"

    print("Splitting the research plan into substasks...")
    print("MODEL: ", MODEL_ID)
    print("PROVIDER: ", PROVIDER)

    # make api call
    client = InferenceClient(
        api_key=os.environ["HF_TOKEN"],
        provider=PROVIDER,
    )

    completion = client.chat_completion(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": TASK_SPLITTER_SYSTEM_INSTRUCTIONS},
            {"role": "user", "content": research_plan},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": TASK_SPLITTER_JSON_SCHEMA, # format
        }
    )

    print("RAW MESSAGE:")
    msg = completion.choices[0].message
    print(json.loads(msg.content))

    message = completion.choices[0].message

    subtasks = json.loads(message.content)['subtasks']

    print("\033[93mGenerated The Following Subtasks\033[0m")
    for task in subtasks:
        print(f"\033[93m{task['title']}\033[0m")
        pprint(f"\033[93m{task['description']}\033[0m")
        print()

    # print("Generated The Following Subtasks")
    # for task in subtasks:
    #     print(task["title"])
    #     print(task["description"])
    #     print()

    return subtasks

subtasks = split_into_subtasks(research_plan)

Splitting the research plan into substasks...
MODEL:  openai/gpt-oss-120b
PROVIDER:  together
RAW MESSAGE:
{'subtasks': [{'id': 'baseline_data', 'title': 'Collect and Process Baseline Climate Normals and Historical Records', 'description': 'Gather 30‑year climate normals (1991‑2020) for the selected regions (Hauts‑de‑France, Normandy, Brittany) and representative cities (Lille, Rouen, Caen, Brest, Amiens) from Météo‑France official datasets. Where city‑specific stations are missing, substitute the nearest Météo‑France RADOME station. Retrieve monthly normals for temperature, precipitation, sunshine hours, wind speed/direction, and relative humidity; record extreme values (record high/low temperature, maximum daily precipitation, maximum wind gust). Acquire the earlier 1961‑1990 normals for comparison. Extract additional parameters: frost days per year, growing season length, number of rainy days (>1\u202fmm), snowfall frequency, and Köppen‑Geiger classification for each location. Docum

In [5]:
subtasks

[{'id': 'baseline_data',
  'title': 'Collect and Process Baseline Climate Normals and Historical Records',
  'description': 'Gather 30‑year climate normals (1991‑2020) for the selected regions (Hauts‑de‑France, Normandy, Brittany) and representative cities (Lille, Rouen, Caen, Brest, Amiens) from Météo‑France official datasets. Where city‑specific stations are missing, substitute the nearest Météo‑France RADOME station. Retrieve monthly normals for temperature, precipitation, sunshine hours, wind speed/direction, and relative humidity; record extreme values (record high/low temperature, maximum daily precipitation, maximum wind gust). Acquire the earlier 1961‑1990 normals for comparison. Extract additional parameters: frost days per year, growing season length, number of rainy days (>1\u202fmm), snowfall frequency, and Köppen‑Geiger classification for each location. Document dataset versions, download dates, and any data gaps. Also pull the same variables from the EU Copernicus ERA5‑La

# Research Coordinator

In [6]:
from smolagents import LiteLLMModel, ToolCallingAgent, MCPClient, tool, InferenceClientModel
import os

SUBAGENT_PROMPT_TEMPLATE = """
You are a specialized research sub-agent.

Global user query:
{user_query}

Overall research plan:
{research_plan}

Your specific subtask (ID: {subtask_id}, Title: {subtask_title}) is:

\"\"\"{subtask_description}\"\"\"

Instructions:
- Focus ONLY on this subtask, but keep the global query in mind for context.
- Use the available tools to search for up-to-date, high-quality sources.
- Prioritize primary and official sources when possible.
- Be explicit about uncertainties, disagreements in the literature, and gaps.
- Return your results as a MARKDOWN report with this structure:

# [Subtask ID] [Subtask Title]

## Summary
Short overview of the main findings.

## Detailed Analysis
Well-structured explanation with subsections as needed.

## Key Points
- Bullet point
- Bullet point

## Sources
- [Title](url) - short comment on why this source is relevant

Now perform the research and return ONLY the markdown report.
"""

COORDINATOR_PROMPT_TEMPLATE = """
You are the LEAD RESEARCH COORDINATOR AGENT.

The user has asked:
\"\"\"{user_query}\"\"\"

A detailed research plan has already been created:

\"\"\"{research_plan}\"\"\"

This plan has been split into the following subtasks (JSON):

```json
{subtasks_json}
```
Each element has the shape:
{{
“id”: “timeframe_confirmation”,
“title”: “Confirm Research Scope Parameters”,
“description”: “Analyze the scope parameters…”
}}

You have access to a tool called:
• initialize_subagent(subtask_id: str, subtask_title: str, subtask_description: str)

Your job:
1. For EACH subtask in the JSON array, call initialize_subagent exactly once
with:
• subtask_id       = subtask[“id”]
• subtask_title    = subtask[“title”]
• subtask_description = subtask[“description”]
2. Wait for all sub-agent reports to come back. Each tool call returns a
markdown report for that subtask.
3. After you have results for ALL subtasks, synthesize them into a SINGLE,
coherent, deeply researched report addressing the original user query
("{user_query}").

Final report requirements:
• Integrate all sub-agent findings; avoid redundancy.
• Make the structure clear with headings and subheadings.
• Highlight:
• key drivers and mechanisms of insecurity,
• historical and temporal evolution,
• geographic and thematic patterns,
• state capacity, public perception, and socioeconomic correlates,
• open questions and uncertainties.
• Include final sections:
• Open Questions and Further Research
• Bibliography / Sources: merge and deduplicate the key sources from all sub-agents.

Important:
• DO NOT expose internal tool-call mechanics to the user.
• Your final answer to the user should be a polished markdown report.
"""

FIRECRAWL_API_KEY = os.environ["FIRECRAWL_API_KEY"]
MCP_URL = f"https://mcp.firecrawl.dev/{FIRECRAWL_API_KEY}/v2/mcp"

# You can vary models here:
COORDINATOR_MODEL_ID = "meta-llama/Llama-4-Scout-17B-16E-Instruct"
SUBAGENT_MODEL_ID    = "meta-llama/Llama-4-Scout-17B-16E-Instruct"

def run_deep_research(user_query: str) -> str:
    print("Running the deep research...")

    # 1) Generate research plan
    research_plan = generate_research_plan(user_query)

    # 2) Split into explicit subtasks
    subtasks = split_into_subtasks(research_plan)

    # 3) Coordinator + sub-agents, all sharing the Firecrawl MCP tools
    print("Initializing Coordinator")
    print("Coordinator Model: ", COORDINATOR_MODEL_ID)
    print("Subagent Model: ", SUBAGENT_MODEL_ID)

    coordinator_model = InferenceClientModel(
        model_id=COORDINATOR_MODEL_ID,
        api_key=os.environ["HF_TOKEN"],
        provider="nscale",
        bill_to="huggingface"
        )
    subagent_model = InferenceClientModel(
        model_id=SUBAGENT_MODEL_ID,
        api_key=os.environ["HF_TOKEN"],
        provider="nscale",
        bill_to="huggingface"
        )

    with MCPClient({"url": MCP_URL, "transport": "streamable-http"}) as mcp_tools:

        # ---- Initialize Subagent TOOL --------------------------------------
        @tool
        def initialize_subagent(subtask_id: str, subtask_title: str, subtask_description: str) -> str:
            """
           Spawn a dedicated research sub-agent for a single subtask.

            Args:
                subtask_id (str): The unique identifier for the subtask.
                subtask_title (str): The descriptive title of the subtask.
                subtask_description (str): Detailed instructions for the sub-agent to perform the subtask.

            The sub-agent:
            - Has access to the Firecrawl MCP tools.
            - Must perform deep research ONLY on this subtask.
            - Returns a structured markdown report with:
              - a clear heading identifying the subtask,
              - a narrative explanation,
              - bullet-point key findings,
              - explicit citations / links to sources.
            """
            print(f"Initializing Subagent for task {subtask_id}...")

            subagent = ToolCallingAgent(
                tools=mcp_tools,                # Firecrawl MCP toolkit
                model=subagent_model,
                add_base_tools=False,
                name=f"subagent_{subtask_id}",
            )

            subagent_prompt = SUBAGENT_PROMPT_TEMPLATE.format(
                user_query=user_query,
                research_plan=research_plan,
                subtask_id=subtask_id,
                subtask_title=subtask_title,
                subtask_description=subtask_description,
            )

            return subagent.run(subagent_prompt)

        # ---- Coordinator agent ---------------------------------------------
        coordinator = ToolCallingAgent(
            tools=[initialize_subagent],
            model=coordinator_model,
            add_base_tools=False,
            name="coordinator_agent",
        )

        # Coordinator prompt: it gets the list of subtasks and the tool
        subtasks_json = json.dumps(subtasks, indent=2, ensure_ascii=False)

        coordinator_prompt = COORDINATOR_PROMPT_TEMPLATE.format(
            user_query=user_query,
            research_plan=research_plan,
            subtasks_json=subtasks_json,
        )

        final_report = coordinator.run(coordinator_prompt)
        return final_report

In [ ]:
result = run_deep_research("Resarch the climate in northern france")

Running the deep research...
Generating the research plan for the query:  Resarch the climate in northern france
MODEL:  moonshotai/Kimi-K2-Thinking
PROVIDER:  auto
